kernel : Python (Pixi)

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import anndata as ad
import numpy as np
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "gao"
MOUSE_SERIES = "gao_mouse"
HUMAN_SERIES = "gao_human"
OLIGO_MOUSE = MOUSE_SERIES + "_oligo"
OLIGO_HUMAN = HUMAN_SERIES + "_oligo"

clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

oligo_mouse = ad.read_h5ad(os.path.join(clustered_h5ad_dir, OLIGO_MOUSE + ".h5ad"))
oligo_human = ad.read_h5ad(os.path.join(clustered_h5ad_dir, OLIGO_HUMAN + ".h5ad"))

In [36]:
conditions = [
    oligo_mouse.obs["sample"] == "SAMN29625768",
    oligo_mouse.obs["sample"] == "SAMN29625767",
    oligo_mouse.obs["sample"] == "SAMN29625766",
    oligo_mouse.obs["sample"] == "SAMN29625765",

    oligo_mouse.obs["sample"] == "SAMN29625764",
    oligo_mouse.obs["sample"] == "SAMN29625763",
    oligo_mouse.obs["sample"] == "SAMN29625762",
    oligo_mouse.obs["sample"] == "SAMN29625761",
]

choices = [
    "4_week",
    "4_week",
    "4_week",
    "4_week",

    "90_week",
    "90_week",
    "90_week",
    "90_week",
]
oligo_mouse.obs["age"] = np.select(conditions, choices, default="unknown")

In [ ]:
cell_marker_genes = {
    "OPC": [ "PDGFRA", "SEZ6L", "KCNIP1", "TMEM132C", "NTNG1", "NKAIN3", "CSPG5", "SCN1A", "LAMA4", "PTPRZ1", "LHFPL3", "PCDH15", "NXPH1", ],
    "COP": ["SOX6", "UST", "BRINP1", "SH3RF3", "GPC3", "GPR17", "WASF1", "BCAS1", "EPB41L2", "ENPP6", "RAPGEF1", "EPHB1", "TCF7L2", "PRICKLE1", ],
    "NFOL": ["FRY", "GOLIM4", "DOCK6", "PIP5K1B", "ABLIM2", "RGS16", "PDGFA", "NREP", "GLRA3", "CMTM7", "TMTC1", "MAN1A1", ],
    "MOL1": ["LAMA2", "OPALIN", "SVEP1", "ROR1", "LTBP1", "DYSF", "PLXDC2", ],
    "MOL2": ["SLC5A11", "ACSBG1", "ACTN2", "HIVEP3", "ANKRD18A", "DPYSL5", "LRRC63", "MOBP", "MAG", "MBP", "PDE1A", "RNF220", "NKAIN2", ],
}

plot.plot_dotplot(oligo_human, OLIGO_HUMAN, cell_marker_genes, group = "cell")

In [ ]:
cell_marker_genes = {
    "Query": ["Mef2a", "Igsf11", "Tcp11l2", "Eml2",  "Lama2", "Svep1", "Ror1", "Ltbp1", "Dysf", "Plxdc2", 
              "Slc5a11", "Acsbg1", "Actn2", "Hivep3", "Dpysl5", "Lrrc63", "Pcsk6",  ],
    "OPC": ["Pdgfra", "Cdo1", "Fabp7", "Hes5", "Bmper", "Rgcc", "Dgkb", "Sgcz", "Chl1", "Ptpn5", "Rlbp1", "Unc13c", "Cspg5", "Calb2", "Pnlip"],
    "COP": ["Sox6", "Gpr17", "Neu4", "Bmp4", "Mycl", "Ust", "Epb41l2", "Enpp6", "Map1b" ],
    "NFOL": ["9630013A20Rik", "Tcf7l2", "Rasgef1b", "Frmd4a", "Pik3r3", "Pdgfa", "Peli1", "Dock6", "Arpc1b", "Ddc", "Rgs16", "Dbx2", "Cmtm7", "Gsto1", ],
    "MFOL": ["Man1a", "Bmp1", "Mgat5", "Ctps1", "Jcad", "Arhgef19", "Btbd16", "Nherf2", "Cxcl12", "Opalin" ], 
    "MOL": ["Car2", "Apod", "Hapln2", "Anln", "Spock3", "Rasgrp3", "Serpinb1a", "Il33", "Mog", "Mobp", "Mag", "Cnp", "Myrf", "Mbp", ],
}

plot.plot_dotplot(oligo_mouse, OLIGO_MOUSE, cell_marker_genes, group = "cell")

In [56]:
# leiden_to_celltype = {
#     "0": "OPC",
#     "1": "MOL1",
#     "2": "MOL1",
#     "3": "NFOL",
#     "4": "MOL1",
#     "5": "MOL2",
#     "6": "OPC",
#     "7": "OPC",
#     "8": "MOL2",
#     "9": "MOL2",
#     "10": "COP",
#     "11": "MOL2",
#     "12": "OPC",
#     "13": "OPC",
#     "14": "MOL1",
#     "15": "OPC",
#     "16": "COP",
#     "17": "OPC",
#     "18": "MOL2",
#     "19": "MOL2",
# }

leiden_to_celltype = {
    "0": "OPC",
    "1": "MOL",
    "2": "MOL",
    "3": "NFOL",
    "4": "MOL",
    "5": "MOL",
    "6": "OPC",
    "7": "OPC",
    "8": "MOL",
    "9": "MOL",
    "10": "COP",
    "11": "MOL",
    "12": "OPC",
    "13": "OPC",
    "14": "MOL",
    "15": "OPC",
    "16": "COP",
    "17": "OPC",
    "18": "MOL",
    "19": "MOL",
}

leiden_str = oligo_human.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
oligo_human.obs["cell"] = mapped

In [3]:
leiden_to_celltype = {
    "0": "MOL",
    "1": "MOL",
    "2": "MOL",
    "3": "MOL",
    "4": "MOL",
    "5": "MOL",
    "6": "MOL",
    "7": "MOL",
    "8": "MOL",
    "9": "MOL",
    "10": "MOL",
    "11": "MOL",
    "12": "MOL",
    "13": "MOL",
    "14": "MOL",
    "15": "MOL",
    "16": "MOL",
    "17": "MOL",
    "18": "MOL",
    "19": "MOL",
    "20": "MFOL",
    "21": "OPC",
    "22": "MFOL",
    "23": "MOL",
    "24": "OPC",
    "25": "OPC",
    "26": "MOL",
    "27": "MOL",
    "28": "COP",
    "29": "COP",
    "30": "NFOL",
    "31": "NFOL",
}

leiden_str = oligo_mouse.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
oligo_mouse.obs["cell"] = mapped

In [57]:
oligo_human.obs["celltype"] = oligo_human.obs["cell"].copy() #type: ignore
plot.plot_umap(oligo_human, OLIGO_HUMAN, has_celltype=True)

In [7]:
oligo_mouse.obs["celltype"] = oligo_mouse.obs["cell"].copy() #type: ignore
plot.plot_umap(oligo_mouse, OLIGO_MOUSE, has_celltype=True)

In [61]:
plot.plot_violin(oligo_human, OLIGO_HUMAN, gene=["IGSF11", "PDGFRA", "CSPG4", "BCAS1", "PDGFA", "ENPP6", "OPALIN", "MOBP"], group="cell", group_order=["OPC", "COP", "NFOL", "MOL"])

In [60]:
plot.plot_violin(oligo_mouse, OLIGO_MOUSE, gene=["Igsf11", "Pdgfra", "Cspg4", "Bcas1", "Pdgfa", "Enpp6", "Opalin", "Mobp"], group="cell", group_order=["OPC", "COP", "NFOL", "MFOL", "MOL"])

In [47]:
plot.plot_proportions(oligo_mouse, OLIGO_MOUSE, group_key="age", group_order=["4_week", "90_week"], sample_key="cell", sample_order=["OPC", "COP", "NFOL", "MFOL", "MOL"], connect=True)
plot.plot_proportions(oligo_mouse, OLIGO_MOUSE, sample_key="age", sample_order=["4_week", "90_week"], group_key="cell", group_order=["OPC", "COP", "NFOL", "MFOL", "MOL"], connect=True)

In [58]:
plot.plot_proportions(oligo_human, OLIGO_HUMAN, group_key="agegroup", group_order=["Infancy", "Childhood", "Adolescence", "Adult", "Aging"], sample_key="cell", sample_order=["OPC", "COP", "NFOL", "MOL", ], connect=True)
plot.plot_proportions(oligo_human, OLIGO_HUMAN, sample_key="agegroup", sample_order=["Infancy", "Childhood", "Adolescence", "Adult", "Aging"], group_key="cell", group_order=["OPC", "COP", "NFOL", "MOL", ], connect=True)

In [65]:
from pipeline.utils.glmmseq import pseudobulk_glmmseq_comp
import numpy as np

deg_mouse = [
    oligo_mouse.obs["cell"] == "NFOL",
    oligo_mouse.obs["cell"] == "OPC",
    oligo_mouse.obs["cell"] == "COP",
    oligo_mouse.obs["cell"] == "MFOL",
    oligo_mouse.obs["cell"] == "MOL"
]
deg_human = [
    oligo_human.obs["cell"] == "NFOL",
    oligo_human.obs["cell"] == "OPC",
    oligo_human.obs["cell"] == "COP",
    oligo_human.obs["cell"] == "MFOL",
    oligo_human.obs["cell"] == "MOL"
]

choices = [
    "NFOL",
    "REST",
    "REST",
    "REST",
    "REST"
]

oligo_mouse.obs["deg"] = np.select(deg_mouse, choices, default="unknown")
oligo_human.obs["deg"] = np.select(deg_human, choices, default="unknown")
print(oligo_mouse.obs["deg"].value_counts())
print(oligo_human.obs["deg"].value_counts())

deg
REST    21892
NFOL      281
Name: count, dtype: int64
deg
REST    11615
NFOL      318
Name: count, dtype: int64


In [66]:
mouse_deg = pseudobulk_glmmseq_comp(oligo_mouse, "deg", group_test = "NFOL", group_control = "REST", random_effect_col= "donor", min_cells=15)

In [71]:
plot.plot_volcano(mouse_deg, OLIGO_MOUSE, "NFOL_vs_REST", genes=["Igsf11", "Pdgfra", "Cspg4", "Bcas1", "Pdgfa", "Enpp6", "Opalin", "Mobp"], xlim=(-7, 7), ylim=(0, 25),)

In [72]:
human_deg = pseudobulk_glmmseq_comp(oligo_human, "deg", group_test = "NFOL", group_control = "REST", random_effect_col= "donor", min_cells=15)

In [73]:
plot.plot_volcano(human_deg, OLIGO_HUMAN, "NFOL_vs_REST", genes=["IGSF11", "PDGFRA", "CSPG4", "BCAS1", "PDGFA", "ENPP6", "OPALIN", "MOBP"], xlim=(-7, 7), ylim=(0, 25),)

In [74]:
from pipeline.utils.enrichment import gsea

mouse_gsea = gsea(mouse_deg, species="mouse", series=OLIGO_MOUSE, libraries=["GO_Biological_Process_2026", "KEGG_2026"])

2026-09-28 22:30:27,375 [WARNING] Duplicated values found in preranked stats: 0.69% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-09-28 22:32:15,479 [WARNING] Duplicated values found in preranked stats: 0.69% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [ ]:
plot.plot_gsea(mouse_gsea, OLIGO_MOUSE, name="GO Biological Process", library="GO_Biological_Process_2026", 
               keywords = ["myelin", "oligodendrocyte", "ensheath", "sterol", "ceramide", "sulfatide", "choline", "plasmalogen", "mbp", "lipid"])

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,Library
0,prerank,Sterol Biosynthetic Process (GO:0016126),0.865290,2.402239,0.0,0.000000,0.0000,20/28,6.48%,Lss;Acly;Fdft1;Mvd;Msmo1;Pmvk;Hmgcs1;Fdps;Sc5d...,GO_Biological_Process_2026
1,prerank,Cholesterol Biosynthetic Process (GO:0006695),0.883144,2.400892,0.0,0.000000,0.0000,20/26,6.48%,Lss;Acly;Fdft1;Mvd;Msmo1;Pmvk;Hmgcs1;Fdps;Sc5d...,GO_Biological_Process_2026
9,prerank,Cholesterol Metabolic Process (GO:0008203),0.675617,2.168654,0.0,0.001106,0.0045,25/60,6.48%,Lss;Acly;Fdft1;Mvd;Cyp46a1;Msmo1;Pmvk;Hmgcs1;F...,GO_Biological_Process_2026


In [86]:
plot.plot_gsea(mouse_gsea, OLIGO_MOUSE, name="KEGG", library="KEGG_2026", 
               keywords = ["myelin", "oligodendrocyte", "differentiat" "ensheath", "sterol", "ceramide", "sulfatide", "choline", "plasmalogen", "mbp", "lipid"])

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,Library
3317,prerank,GLYCOSPHINGOLIPID BIOSYNTHESIS - LACTO AND NEO...,-0.748539,-1.887478,0.000000,0.006174,0.0235,10/20,12.35%,St3gal4;B3galt5;Fut9;B3galt1;St3gal3;B3gnt2;St...,KEGG_2026
3341,prerank,GLYCOSPHINGOLIPID BIOSYNTHESIS - GLOBO AND ISO...,-0.760590,-1.697054,0.006369,0.036098,0.3510,7/12,14.34%,B3galt5;Fut9;St3gal2;St8sia1;B3galnt1;St3gal1;...,KEGG_2026


In [75]:
human_gsea = gsea(human_deg, species="human", series=OLIGO_HUMAN, libraries=["GO_Biological_Process_2026", "KEGG_2026"])

In [87]:
plot.plot_gsea(human_gsea, OLIGO_HUMAN, name="GO Biological Process", library="GO_Biological_Process_2026", 
               keywords = ["myelin", "oligodendrocyte", "differentiat" "ensheath", "sterol", "ceramide", "sulfatide", "choline", "plasmalogen", "mbp", "lipid"])

ValueError: No pathways pass the keyword, direction, and FDR filters.

In [93]:
plot.plot_species_scatter(human_deg, mouse_deg, SERIES, name="NFOL_vs_REST", genes=["IGSF11", "PDGFRA", "CSPG4", "BCAS1", "PDGFA", "ENPP6", "OPALIN", "MOBP"], xlim=(-7, 7), ylim=(-7, 7),)

,human_gene,human_lfc,human_padj,mouse_gene,mouse_lfc,mouse_padj,status
0,A2M,-0.028306,0.970343,A2m,-0.115095,8.866748e-01,padj cutoff not met
1,A2ML1,0.150725,0.849334,A2ml1,0.295726,6.134881e-01,padj cutoff not met
2,AAAS,0.487529,0.302368,Aaas,0.198952,4.937265e-01,padj cutoff not met
3,AACS,0.043305,0.950290,Aacs,1.831848,6.562574e-12,padj cutoff not met
4,AADAT,-0.161812,0.835130,Aadat,0.019631,9.830633e-01,padj cutoff not met
...,...,...,...,...,...,...,...
10936,HOXD1,0.077691,0.896886,Hoxd1,-0.025295,9.778466e-01,padj cutoff not met
10937,MAP6D1,0.117782,0.851374,Map6d1,-0.237218,2.415996e-01,padj cutoff not met
10938,MYO1A,0.049311,0.935895,Myo1a,0.576475,3.272893e-01,padj cutoff not met
10939,TMEM88B,0.230303,0.748446,Tmem88b,-1.313510,2.022881e-05,padj cutoff not met
